In [5]:
import pandas as pd
import plotly.express as px
from pandas.core.frame import DataFrame
from plotly.graph_objs._figure import Figure

res: DataFrame = pd.read_csv("season_17.csv")
long_res: DataFrame = (
    pd.wide_to_long(df=res, stubnames=["Week "], i="Player", j="week")
    .reset_index()
    .rename(columns={"Week ": "points"})
)
long_res: DataFrame = long_res.sort_values(["Player", "week"])
long_res["cumulative_points"] = long_res.groupby(by="Player")["points"].cumsum()

fig: Figure = px.line(
    data_frame=long_res,
    x="week",
    y="cumulative_points",
    color="Player",
    template="simple_white",
    title="Cumulative points by week",
)
fig.update_layout(height=500)
fig.show()


long_res["weekly_mean"] = long_res.groupby("week")["points"].transform("mean")
long_res["cumulative_deviation"] = (
    (long_res["points"] - long_res["weekly_mean"]).groupby(long_res["Player"]).cumsum()
)

fig_deviation: Figure = px.line(
    data_frame=long_res,
    x="week",
    y="cumulative_deviation",
    color="Player",
    template="simple_white",
    title="Cumulative points above/below weekly average",
)
fig_deviation.add_hline(
    y=0,
    line={"color": "black", "width": 3, "dash": "solid"},
    annotation_text="At weekly average",
)
fig_deviation.update_yaxes(range=[-50, 60])
fig_deviation.update_layout(height=800)
fig_deviation.show()